# Conservation Prevents Blowup

This notebook repeats the one-dimensional experiment of the [`Wave1D.ipynb`](./Wave1D.ipynb) notebook with the entries of the parameter vector $\boldsymbol \mu$ sampled from a wider range than before. Additionally, we compare the Hamiltonians of each ROM across all wave equation experiments.

**Objective**: Demonstrate that the non-symmetric OpInf ROM may produce un-physical solutions due to its erroneous energy gain.

 To begin, import a few standard Python scientific libraries, the [`opinf`](https://willcox-research-group.github.io/rom-operator-inference-Python3) package, and a few local files.

In [ ]:
import opinf
import numpy as np
import matplotlib.pyplot as plt

import utils
import models
import waveEq as wave

In [ ]:
utils.matplotlib_config()

If any of these imports fail, see the [README](./README.md) for installation instructions.

## 1D Wave Experiment Revisited

### Training/Testing Data Generation

We start by constructing the finite element model $(3.2)$ from [`Wave1D.ipynb`](./Wave1D.ipynb), also called the full-order model (FOM).

In [ ]:
t = np.linspace(0, 8 * np.pi, 1001)
fom = wave.WaveFEM1D(num_elements=500, orderW=0, orderV=1)
print(fom)

We randomly sample parameters $\boldsymbol{\mu}$, the entries of which are uniformly spaced in $(0.8, 8)$.
The samples are split into distinct training and testing sets.

In [ ]:
training_parameters, testing_parameters = fom.sample_parameters(
    low=0.8,
    high=8,  # Higher parameter range than the other 1D experiment.
    num_samples=50,
    train_ratio=0.80,
    randseed=0,
)

print(f"{training_parameters.shape=}")
print(f"{testing_parameters.shape=}")

We now solve the FOM for each training parameter instance to generate training data, as well as for each testing parameter instance to create a test set to compare to ROM predictions later on.

In [ ]:
training_Q, training_P = fom.solve_multi(training_parameters, t)
testing_Q, testing_P = fom.solve_multi(testing_parameters, t)

training_snapshots = np.concatenate((training_Q, training_P), axis=1)
testing_snapshots = np.concatenate((testing_Q, testing_P), axis=1)

print(f"{training_snapshots.shape=}")
print(f"{testing_snapshots.shape=}")

### Reduced-order Models

Next, we compute a PSD basis matrix and learn reduced-order models for $(4.3)$ from [`Wave1D.ipynb`](./Wave1D.ipynb) through symmetric (H-OpInf) and non-symmetric (OpInf) operator inference.

In [ ]:
# Extract the mass matrix.
M = fom.MW.toarray()
print(f"Mass matrix: {type(M)=}, {M.shape=}")

# Compute the PSD basis diag(U, U) such that U^T M U is the identity.
basis = models.PSDBasis(num_vectors=30, weights=M).fit(
    np.hstack(training_snapshots)
)
print(basis)

In [ ]:
ddter = opinf.ddt.UniformFiniteDifferencer(t, "ord4")

opinf_rom_sym = opinf.ParametricROM(
    basis=basis,
    ddt_estimator=ddter,
    model=models.BlockHamiltonianTensorModel(
        fom.parameter_dimension,
        symmetric=True,
    ),
).fit(training_parameters, training_snapshots, fit_basis=False)

opinf_rom_nosym = opinf.ParametricROM(
    basis=opinf_rom_sym.basis,
    ddt_estimator=ddter,
    model=models.BlockHamiltonianTensorModel(
        fom.parameter_dimension,
        symmetric=False,  # No symmetry constraint
    ),
).fit(training_parameters, training_snapshots, fit_basis=False)

### Trajectories

We now solve the the two ROMs constructed above, as well as the intrusive Galerkin ROM, as a single training parameter and plot the solutions together at several time instances.

In [ ]:
idx = 13
mu = training_parameters[idx]
Q_fom = training_Q[idx]
y0 = training_snapshots[idx][:, 0]

# Intrusive Galerkin ROM.
Q_intrusive, P_intrusive = fom.solveROM(basis.pod, mu, t)

# Symmetric Hamiltonian operator inference ROM.
Y_symOpInf = opinf_rom_sym.predict(mu, y0, t)
Q_symOpInf, P_symOpInf = np.split(Y_symOpInf, 2, axis=0)

# Standard, non-symmetric operator inference ROM.
Y_nosymOpInf = opinf_rom_nosym.predict(mu, y0, t)
Q_nosymOpInf, P_nosymOpInf = np.split(Y_nosymOpInf, 2, axis=0)

In [ ]:
times = np.array([0, len(t) // 3, 2 * len(t) // 3, len(t) - 1])
time_labels = [r"$t=0$", r"$t=8\pi/3$", r"$t=16\pi/3$", r"$t=8\pi$"]

fig, axs = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True)
axs = axs.flatten()
xdata = fom.nodes

for ax, frame, label in zip(axs, times, time_labels):
    ax.plot(xdata, Q_fom[:, frame], "k-", label="FOM")
    ax.plot(xdata, Q_intrusive[:, frame], "C2--", label="Intrusive")
    ax.plot(xdata, Q_symOpInf[:, frame], "C0-.", label="H-OpInf")
    ax.plot(xdata, Q_nosymOpInf[:, frame], "C1:", label="OpInf")
    # markerfacecolor="none",

    ax.set_ylim([-0.5, 0.5])
    ax.set_xlim([0, 2 * np.pi])
    ax.annotate(label, xy=(0.05, 0.8), xycoords="axes fraction", fontsize=20)

axs[0].set_ylabel(r"Position $q$")
axs[2].set_ylabel(r"Position $q$")
axs[2].set_xlabel(r"space $x$")
axs[3].set_xlabel(r"space $x$")

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=4,
    framealpha=0,
    fontsize="x-large",
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.175)
plt.show()

Both the intrusive Galerkin ROM and the symmetric H-OpInf ROM produce trajectories that resemble the true solution and which remain stable for all time. In contrast, the non-symmetric OpInf ROM yields an un-physical trajectory that eventually blows up due to a persistent energy gain.

## Hamiltonian Conservation

Finally, we compute and plot the reduced Hamiltonian $(3.5)$ from [`Wave1D.ipynb`](./Wave1D.ipynb) for the three ROM generated trajectories, and show that it is not conserved by the non-symmetric ROM. 

The Hamiltonian plots in the other two panels of Figure 4.8 in the paper, corresponding to the experiments in [`Wave1D.ipynb`](./Wave1D.ipynb) and [`Wave2D.ipynb`](./Wave2D.ipynb), are also generated by the code in the blocks that follow.

In [ ]:
# Intrusive Galerkin ROM.
H_FOM = fom.reduced_Hamiltonian(basis.pod, Q_intrusive, P_intrusive, mu)

# Symmetric Hamiltonian operator inference ROM.
H_symOpInf = opinf_rom_sym.model.Hamiltonian(
    opinf_rom_sym.encode(Y_symOpInf), mu
)

# Standard, non-symmetric operator inference ROM.
H_nosymOpInf = opinf_rom_nosym.model.Hamiltonian(
    opinf_rom_sym.encode(Y_nosymOpInf), mu
)

In [ ]:
filelabels = ["intrusive", "H-OpInf", "OpInf"]
styles_labels = [("C2-", "Intrusive"), ("C0-.", "H-OpInf"), ("C1:", "OpInf")]

try:
    arr1 = np.column_stack(
        [
            utils.loadnpy(f"wave1D_reducedHamiltonian_{label}.npy")
            for label in filelabels
        ]
    )
    arr3 = np.column_stack(
        [
            utils.loadnpy(f"wave2D_reducedHamiltonian_{label}.npy")
            for label in filelabels
        ]
    )
except FileNotFoundError as e:
    raise RuntimeError(
        ".npy files missing: run other wave notebooks first"
    ) from e

arr2 = np.column_stack([H_FOM, H_symOpInf, H_nosymOpInf])

arrays = [
    np.abs(arr1 - arr1[0, :]),
    np.abs(arr2 - arr2[0, :]),
    np.abs(arr3 - arr3[0, :]),
]

fig, ax = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
ax[0].set_ylim([1e-16, 1e5])
ax[0].set_ylabel(
    r"$|\hat H(\hat{\mathbf{y}}(t),\boldsymbol{\mu})"
    r"-\hat H(\hat{\mathbf{y}}(0),\boldsymbol{\mu})|$",
    fontsize=22,
)

for j in range(len(ax)):
    [
        ax[j].semilogy(arrays[j][:, i], style, label=label, lw=2)
        for i, (style, label) in enumerate(styles_labels)
    ]
    ax[j].set_xlabel(r"time $t$", fontsize=26)
    ax[j].tick_params(labelsize=24)


ax[0].set_xlim([1, 1000])
ax[2].set_xlim([1, 500])
ax[1].set_xlim([1, 1000])

ax[0].set_xticks([1, 250, 500, 750])
ax[0].set_xticklabels(["0", r"$2\pi$", r"$4\pi$", r"$6\pi$"])
ax[1].set_xticks([1, 250, 500, 750])
ax[1].set_xticklabels(["0", r"$2\pi$", r"$4\pi$", r"$6\pi$"])
ax[2].set_xticks([1, 125, 250, 375])
ax[2].set_xticklabels(["0", r"$\pi$", r"$2\pi$", r"$3\pi$"])
ax[0].legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
    ncol=3,
    fontsize=28,
)

fig.tight_layout()
fig.subplots_adjust(bottom=0.35, wspace=0.06)

plt.show()